# RFM (Recency, Frequency, Monetary) Analysis — SQL

Building the RFM base table and supporting queries against the `transactions` table in Postgres. Python 
 used to connect and pull results into a dataframe for a quick preview.

In [14]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv(dotenv_path='../.env')

username = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
database = os.getenv('DB_NAME')

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

### RFM query
 
For each Customer ID:
 - Recency: days since most recent order 
 - Frequency: number of distict orders
 - Monetary: total revenue generated

In [15]:
rfm_query = text("""
WITH last_date AS (
    SELECT MAX("InvoiceDate") AS max_date FROM customer
),
customer_agg AS (
    SELECT
        "Customer ID" AS customer_id,
        MAX("InvoiceDate") AS last_purchase_date,
        COUNT(DISTINCT "Invoice") AS frequency,
        SUM("Revenue") AS monetary
    FROM customer
    WHERE is_cancellation = FALSE
    GROUP BY "Customer ID"
)
SELECT
    customer_id,
    EXTRACT(DAY FROM (SELECT max_date FROM last_date) - last_purchase_date) AS recency_days,
    frequency,
    monetary
FROM customer_agg;
""")

rfm_df = pd.read_sql(rfm_query, engine)
rfm_df.head()

,customer_id,recency_days,frequency,monetary
0,12346.0,325.0,12,77556.46
1,12347.0,1.0,8,4921.53
2,12348.0,74.0,5,2019.40
3,12349.0,18.0,4,4428.69
4,12350.0,309.0,1,334.40


### Segment scoring with NTILE

Customers split into quartiles for each RFM dimension using `NTILE(4)`

In [16]:
rfm_scored_query = text("""
WITH last_date AS (
    SELECT MAX("InvoiceDate") AS max_date FROM customer
),
customer_agg AS (
    SELECT
        "Customer ID" AS customer_id,
        MAX("InvoiceDate") AS last_purchase_date,
        COUNT(DISTINCT "Invoice") AS frequency,
        SUM("Revenue") AS monetary
    FROM customer
    WHERE is_cancellation = FALSE
    GROUP BY "Customer ID"
),
rfm_base AS (
    SELECT
        customer_id,
        EXTRACT(DAY FROM (SELECT max_date FROM last_date) - last_purchase_date) AS recency_days,
        frequency,
        monetary
    FROM customer_agg
)
SELECT
    customer_id,
    recency_days,
    frequency,
    monetary,
    NTILE(4) OVER (ORDER BY recency_days DESC) AS recency_score,
    NTILE(4) OVER (ORDER BY frequency ASC) AS frequency_score,
    NTILE(4) OVER (ORDER BY monetary ASC) AS monetary_score
FROM rfm_base;
""")

rfm_scored = pd.read_sql(rfm_scored_query, engine)
rfm_scored['rfm_total'] = rfm_scored['recency_score'] + rfm_scored['frequency_score'] + rfm_scored['monetary_score']
rfm_scored.head()

,customer_id,recency_days,frequency,monetary,recency_score,frequency_score,monetary_score,rfm_total
0,12636.0,738.0,1,141.00,1,1,1,3
1,17592.0,738.0,1,148.30,1,1,1,3
2,17087.0,737.0,1,221.53,1,1,1,3
3,15833.0,737.0,1,80.40,1,1,1,3
4,13526.0,737.0,2,1182.00,1,2,3,6


### RFM output saved to table

Scored RFM data written back to Potgres for future queries and other notebook access

In [17]:
rfm_scored.to_sql('rfm_scores', engine, if_exists='replace', index=False)
print("rfm_scores table created.")

rfm_scores table created.


### Cancellations rate by product categories

In [18]:
cancellation_query = text("""
SELECT
    "Description" AS product,
    COUNT(*) FILTER (WHERE is_cancellation) AS cancelled_lines,
    COUNT(*) AS total_lines,
    ROUND(
        COUNT(*) FILTER (WHERE is_cancellation)::numeric / COUNT(*) * 100, 2
    ) AS cancellation_rate_pct
FROM customer
GROUP BY "Description"
HAVING COUNT(*) > 50  -- filter out low-volume noise
ORDER BY cancellation_rate_pct DESC
LIMIT 20;
""")

cancellation_df = pd.read_sql(cancellation_query, engine)
cancellation_df

,product,cancelled_lines,total_lines,cancellation_rate_pct
0,Discount,165,170,97.06
1,WHITE CHERRY LIGHTS,119,217,54.84
2,BLACK CHERRY LIGHTS,50,98,51.02
3,Manual,397,1078,36.83
4,GREEN CHERRY LIGHTS,37,113,32.74
5,GOLD CHERRY LIGHTS,24,75,32.00
6,LIGHT PINK CHERRY LIGHTS,60,219,27.40
7,WILLOW BRANCH LIGHTS.,24,88,27.27
8,PINK CHERRY LIGHTS,86,316,27.22
9,CLEAR MILKSHAKE GLASS,16,59,27.12


### Monthly revenue

In [20]:
monthly_revenue_query = text("""
SELECT
    DATE_TRUNC('month', "InvoiceDate") AS month,
    SUM("Revenue") AS total_revenue,
    COUNT(DISTINCT "Invoice") AS order_count
FROM customer
WHERE is_cancellation = FALSE
GROUP BY 1
ORDER BY 1;
""")

monthly_revenue_df = pd.read_sql(monthly_revenue_query, engine)
monthly_revenue_df

,month,total_revenue,order_count
0,2009-12-01,683504.010,1512
1,2010-01-01,555802.672,1011
2,2010-02-01,504558.956,1104
3,2010-03-01,696978.471,1524
4,2010-04-01,591982.002,1329
5,2010-05-01,597833.380,1377
6,2010-06-01,636371.130,1497
7,2010-07-01,589736.170,1381
8,2010-08-01,602224.600,1293
9,2010-09-01,829013.951,1689


In [21]:
# Export monthly revenue and cancellation data for dashboard use
monthly_revenue_df.to_csv('../outputs/monthly_revenue.csv', index=False)
cancellation_df.to_csv('../outputs/cancellation_by_product.csv', index=False)

## Findings summary

- **Cancellation rate:** 
    - "Discount" and "Manual" line items are non-product adjustment 
  entries and were excluded from the product-level view. 
    - Decorative lighting SKUs (Cherry Lights variants) show the highest cancellation rates (27-55%)
- **Seasonality:** 
    - Revenue peaks sharply in October-November both years (pre-holiday ordering), and bottoms out in February. 